# Model Selection Evaluation

Recovered workflow. See `docs/WORKFLOW_ORDER.md` for dependencies and execution restrictions. Source cell numbers below are zero-based.


Source cell: 88.


In [ ]:
# ============================================================
# MODEL-SELECTION / CHECKPOINT-SELECTION SENSITIVITY
# Morgan + RDKit | Seeds 42, 43, 44
# ============================================================
#
# CONTROLLED FACTOR:
#   CHECKPOINT-SELECTION CRITERION ONLY
#
# Saved from the SAME training trajectory:
#   1) Best Calibration Macro-F1
#   2) Best Calibration Accuracy
#   3) Best Calibration Weighted-F1
#
# FIXED:
# - Morgan + RDKit Train-only features = 2249
# - Exact Train / Calibration / Test splits
# - 84 Interaction classes
# - Ordered Drug1 -> Drug2
# - Same 268,244-edge Train graph
# - Same GCN architecture
# - Same weighted CrossEntropyLoss
# - Same optimizer / scheduler
# - Same Macro-F1 early stopping
# - Test evaluated ONLY after training/model selection
#
# ============================================================


# ============================================================
# 1) IMPORTS
# ============================================================

import os
import gc
import json
import random
import hashlib
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F

from sklearn.metrics import (
    accuracy_score,
    f1_score,
)

from torch_geometric.nn import GCNConv


# ============================================================
# 2) CONFIG
# ============================================================

SEEDS = [42, 43, 44]

NUM_CLASSES = 84

INPUT_DIM = 2249

BATCH_SIZE = 512

MAX_EPOCHS = 120

EARLY_STOP_PATIENCE = 18

LR = 7e-4

WEIGHT_DECAY = 1e-4

DROPOUT = 0.35


ROOT = Path(
    "/kaggle/input/datasets/"
    "amiramohammeddd/ddi-v2-menna-projectt"
)


FEATURE_ROOT = Path(
    "/kaggle/working/"
    "FEATURE_ABLATION_TRAIN_ONLY_PREPROCESSING"
)


FEATURE_FILE = (
    FEATURE_ROOT
    /
    "molecular_features_morgan_rdkit.npz"
)


TRAIN_FILE = (
    ROOT
    /
    "clean_train_split.csv"
)

CAL_FILE = (
    ROOT
    /
    "clean_calibration_split.csv"
)

TEST_FILE = (
    ROOT
    /
    "clean_test_split.csv"
)


OUT_ROOT = Path(
    "/kaggle/working/"
    "MODEL_SELECTION_SENSITIVITY_MORGAN_RDKIT_V1"
)


OUT_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


# ============================================================
# 3) HASH HELPER
# ============================================================

def sha256_file(
    path,
    block_size=1024 * 1024
):

    h = hashlib.sha256()

    with open(
        path,
        "rb"
    ) as f:

        while True:

            block = f.read(
                block_size
            )

            if not block:
                break

            h.update(
                block
            )

    return h.hexdigest()


# ============================================================
# 4) RANDOM SEED HELPERS
# ============================================================

def set_seed(seed):

    random.seed(
        seed
    )

    np.random.seed(
        seed
    )

    torch.manual_seed(
        seed
    )

    torch.cuda.manual_seed_all(
        seed
    )

    torch.backends.cudnn.deterministic = True

    torch.backends.cudnn.benchmark = False


def get_rng_state():

    state = {

        "python":
            random.getstate(),

        "numpy":
            np.random.get_state(),

        "torch":
            torch.get_rng_state(),

        "cuda":
            (
                torch.cuda.get_rng_state_all()
                if torch.cuda.is_available()
                else None
            ),
    }

    return state


def set_rng_state(state):

    random.setstate(
        state["python"]
    )

    np.random.set_state(
        state["numpy"]
    )

    torch.set_rng_state(
        state["torch"]
    )

    if (
        torch.cuda.is_available()
        and
        state["cuda"] is not None
    ):

        torch.cuda.set_rng_state_all(
            state["cuda"]
        )


# ============================================================
# 5) VERIFY INPUT FILES
# ============================================================

assert FEATURE_FILE.is_file(), (
    f"Missing feature file:\n{FEATURE_FILE}"
)

assert TRAIN_FILE.is_file()
assert CAL_FILE.is_file()
assert TEST_FILE.is_file()


print("=" * 100)
print("INPUT FILES VERIFIED")
print("=" * 100)

print("Feature cache:")
print(FEATURE_FILE)


# ============================================================
# 6) LOAD SPLITS
# ============================================================

train_df = pd.read_csv(
    TRAIN_FILE
)

cal_df = pd.read_csv(
    CAL_FILE
)

test_df = pd.read_csv(
    TEST_FILE
)


assert len(train_df) == 134249

assert len(cal_df) == 28768

assert len(test_df) == 28768


required_columns = {
    "Drug1",
    "Drug2",
    "Label",
    "label_id",
}


for name, df in [

    (
        "Train",
        train_df
    ),

    (
        "Calibration",
        cal_df
    ),

    (
        "Test",
        test_df
    ),

]:

    assert required_columns.issubset(
        df.columns
    ), (
        name,
        df.columns.tolist()
    )


print("\nSplits:")

print(
    "Train:",
    train_df.shape
)

print(
    "Calibration:",
    cal_df.shape
)

print(
    "Test:",
    test_df.shape
)


# ============================================================
# 7) VERIFY 84 INTERACTION IDs / LABELS
# ============================================================

mapping_df = pd.concat(

    [

        train_df[
            [
                "label_id",
                "Label"
            ]
        ],

        cal_df[
            [
                "label_id",
                "Label"
            ]
        ],

        test_df[
            [
                "label_id",
                "Label"
            ]
        ],

    ],

    ignore_index=True

).drop_duplicates()


assert (
    mapping_df
    .groupby(
        "label_id"
    )["Label"]
    .nunique()
    .max()
    ==
    1
)


assert (
    mapping_df
    .groupby(
        "Label"
    )["label_id"]
    .nunique()
    .max()
    ==
    1
)


assert mapping_df[
    "label_id"
].nunique() == 84


assert mapping_df[
    "Label"
].nunique() == 84


label_ids_sorted = sorted(
    mapping_df[
        "label_id"
    ]
    .unique()
    .tolist()
)


label_to_internal = {

    label_id:
        index

    for index, label_id

    in enumerate(
        label_ids_sorted
    )
}


def encode_labels(df):

    encoded = (
        df[
            "label_id"
        ]
        .map(
            label_to_internal
        )
        .to_numpy(
            dtype=np.int64
        )
    )

    assert len(
        np.unique(
            encoded
        )
    ) == 84

    return encoded


y_train = encode_labels(
    train_df
)

y_cal = encode_labels(
    cal_df
)

y_test = encode_labels(
    test_df
)


print(
    "\n 84 Interaction IDs/labels verified."
)


# ============================================================
# 8) LOAD RECOVERED TRAIN-ONLY FEATURES
# ============================================================

with np.load(
    FEATURE_FILE,
    allow_pickle=True
) as feature_data:

    X = (
        feature_data[
            "x"
        ]
        .astype(
            np.float32
        )
    )

    all_drugs = (
        feature_data[
            "all_drugs"
        ]
        .astype(
            str
        )
    )

    saved_train_indices = (
        feature_data[
            "train_drug_indices"
        ]
        .astype(
            np.int64
        )
    )


assert X.shape == (
    1705,
    2249
)

assert len(
    all_drugs
) == 1705

assert len(
    np.unique(
        all_drugs
    )
) == 1705

assert len(
    saved_train_indices
) == 1675

assert np.isfinite(
    X
).all()


print(
    "\nFeature shape:",
    X.shape
)

print(
    "Total molecular SMILES:",
    len(all_drugs)
)


# ============================================================
# 9) MOLECULAR INDEX
# ============================================================

drug_to_global = {

    drug:
        index

    for index, drug

    in enumerate(
        all_drugs
    )
}


for name, df in [

    (
        "Train",
        train_df
    ),

    (
        "Calibration",
        cal_df
    ),

    (
        "Test",
        test_df
    ),

]:

    split_drugs = set(
        df[
            "Drug1"
        ]
        .astype(
            str
        )
    ).union(

        set(
            df[
                "Drug2"
            ]
            .astype(
                str
            )
        )

    )

    missing = (
        split_drugs
        -
        set(
            drug_to_global
        )
    )

    assert not missing, (
        f"{name}: "
        f"{len(missing)} molecules missing"
    )


# ============================================================
# 10) BUILD ORDERED PAIRS
# ============================================================

def make_global_pairs(df):

    d1 = (
        df[
            "Drug1"
        ]
        .astype(
            str
        )
        .map(
            drug_to_global
        )
        .to_numpy(
            dtype=np.int64
        )
    )


    d2 = (
        df[
            "Drug2"
        ]
        .astype(
            str
        )
        .map(
            drug_to_global
        )
        .to_numpy(
            dtype=np.int64
        )
    )


    return np.column_stack(
        [
            d1,
            d2
        ]
    )


train_pairs_global = make_global_pairs(
    train_df
)

cal_pairs_global = make_global_pairs(
    cal_df
)

test_pairs_global = make_global_pairs(
    test_df
)


assert train_pairs_global.shape == (
    134249,
    2
)


# ============================================================
# 11) EXACT TRAIN GRAPH
#
# Raw Train rows
# -> unique ORIGINAL ordered pairs
# -> append reverse
# -> NO second global deduplication
# ============================================================

forward_unique = np.unique(
    train_pairs_global,
    axis=0
)


assert forward_unique.shape == (
    134122,
    2
)


reverse_unique = (
    forward_unique[
        :,
        ::-1
    ]
    .copy()
)


forward_set = {

    (
        int(a),
        int(b)
    )

    for a, b

    in forward_unique
}


reverse_set = {

    (
        int(a),
        int(b)
    )

    for a, b

    in reverse_unique
}


reciprocal_overlap = len(
    forward_set.intersection(
        reverse_set
    )
)


assert reciprocal_overlap == 122


diagnostic_global_unique = np.unique(

    np.vstack(
        [
            forward_unique,
            reverse_unique
        ]
    ),

    axis=0
)


assert diagnostic_global_unique.shape == (
    268122,
    2
)


graph_pairs_global = np.vstack(
    [
        forward_unique,
        reverse_unique
    ]
)


assert graph_pairs_global.shape == (
    268244,
    2
)


print("\n" + "=" * 100)
print("GRAPH QA")
print("=" * 100)

print(
    "Raw Train interaction rows:",
    len(
        train_pairs_global
    )
)

print(
    "Unique ordered Train edges:",
    len(
        forward_unique
    )
)

print(
    "Reciprocal directed overlap:",
    reciprocal_overlap
)

print(
    "Globally unique bidirectional edges "
    "(diagnostic only):",
    len(
        diagnostic_global_unique
    )
)

print(
    "ACTUAL graph edges:",
    len(
        graph_pairs_global
    )
)

print(
    " No second global deduplication."
)


# ============================================================
# 12) TRAIN-ONLY LOCAL NODE INDEX
# ============================================================

train_drug_global = np.unique(
    train_pairs_global.reshape(
        -1
    )
)


assert len(
    train_drug_global
) == 1675


assert np.array_equal(
    np.sort(
        train_drug_global
    ),
    np.sort(
        saved_train_indices
    )
)


global_to_local = {

    int(global_id):
        local_id

    for local_id, global_id

    in enumerate(
        train_drug_global
    )
}


train_pairs_local = np.array(

    [

        [
            global_to_local[
                int(a)
            ],

            global_to_local[
                int(b)
            ]
        ]

        for a, b

        in train_pairs_global
    ],

    dtype=np.int64
)


graph_pairs_local = np.array(

    [

        [
            global_to_local[
                int(a)
            ],

            global_to_local[
                int(b)
            ]
        ]

        for a, b

        in graph_pairs_global
    ],

    dtype=np.int64
)


X_train_local = X[
    train_drug_global
]


assert X_train_local.shape == (
    1675,
    2249
)


# ============================================================
# 13) HELD-OUT DRUG QA
# ============================================================

train_nodes = set(
    train_drug_global.tolist()
)


cal_nodes = set(
    cal_pairs_global.reshape(
        -1
    ).tolist()
)


test_nodes = set(
    test_pairs_global.reshape(
        -1
    ).tolist()
)


cal_only = (
    cal_nodes
    -
    train_nodes
)


test_only = (
    test_nodes
    -
    train_nodes
)


assert len(
    cal_only
) == 14


assert len(
    test_only
) == 20


graph_nodes = set(
    graph_pairs_global.reshape(
        -1
    ).tolist()
)


assert graph_nodes == train_nodes

assert len(
    graph_nodes.intersection(
        cal_only
    )
) == 0

assert len(
    graph_nodes.intersection(
        test_only
    )
) == 0


print(
    "\nTrain graph drugs:",
    len(
        train_nodes
    )
)

print(
    "Calibration unseen drugs:",
    len(
        cal_only
    )
)

print(
    "Test unseen drugs:",
    len(
        test_only
    )
)


# ============================================================
# 14) DEVICE
# ============================================================

device = torch.device(

    "cuda"

    if torch.cuda.is_available()

    else "cpu"
)


print(
    "\nDevice:",
    device
)


# ============================================================
# 15) TORCH TENSORS
# ============================================================

x_all = torch.tensor(
    X,
    dtype=torch.float32,
    device=device
)


x_train = torch.tensor(
    X_train_local,
    dtype=torch.float32,
    device=device
)


edge_train = torch.tensor(
    graph_pairs_local.T,
    dtype=torch.long,
    device=device
)


edge_global = torch.tensor(
    graph_pairs_global.T,
    dtype=torch.long,
    device=device
)


train_pairs_t = torch.tensor(
    train_pairs_local,
    dtype=torch.long,
    device=device
)


cal_pairs_t = torch.tensor(
    cal_pairs_global,
    dtype=torch.long,
    device=device
)


test_pairs_t = torch.tensor(
    test_pairs_global,
    dtype=torch.long,
    device=device
)


y_train_t = torch.tensor(
    y_train,
    dtype=torch.long,
    device=device
)


# ============================================================
# 16) CLASS WEIGHTS
# ============================================================

class_counts = np.bincount(
    y_train,
    minlength=NUM_CLASSES
).astype(
    np.float64
)


assert np.all(
    class_counts > 0
)


max_count = (
    class_counts.max()
)


class_weights = np.sqrt(

    max_count
    /
    (
        class_counts
        +
        1.0
    )
)


class_weights = np.clip(
    class_weights,
    0.5,
    3.0
)


class_weights = (
    class_weights
    /
    class_weights.mean()
)


class_weights_t = torch.tensor(
    class_weights,
    dtype=torch.float32,
    device=device
)


print(
    "\nClass weight range:",
    float(
        class_weights.min()
    ),
    "to",
    float(
        class_weights.max()
    )
)


# ============================================================
# 17) MODEL
# ============================================================

class GCNDrugEncoder(nn.Module):

    def __init__(
        self,
        input_dim,
        dropout=0.35
    ):

        super().__init__()


        self.conv1 = GCNConv(
            input_dim,
            384,
            cached=False,
            add_self_loops=True,
            normalize=True
        )


        self.bn1 = nn.BatchNorm1d(
            384
        )


        self.conv2 = GCNConv(
            384,
            384,
            cached=False,
            add_self_loops=True,
            normalize=True
        )


        self.bn2 = nn.BatchNorm1d(
            384
        )


        self.conv3 = GCNConv(
            384,
            192,
            cached=False,
            add_self_loops=True,
            normalize=True
        )


        self.bn3 = nn.BatchNorm1d(
            192
        )


        self.dropout = nn.Dropout(
            dropout
        )


    def forward(
        self,
        x,
        edge_index
    ):

        x = self.conv1(
            x,
            edge_index
        )

        x = self.bn1(
            x
        )

        x = F.relu(
            x
        )

        x = self.dropout(
            x
        )


        x = self.conv2(
            x,
            edge_index
        )

        x = self.bn2(
            x
        )

        x = F.relu(
            x
        )

        x = self.dropout(
            x
        )


        x = self.conv3(
            x,
            edge_index
        )

        x = self.bn3(
            x
        )

        return x


class FinalConcatOnlyGCN(nn.Module):

    def __init__(
        self,
        input_dim,
        num_classes=84,
        dropout=0.35
    ):

        super().__init__()


        self.encoder = GCNDrugEncoder(
            input_dim=input_dim,
            dropout=dropout
        )


        self.classifier = nn.Sequential(

            nn.LayerNorm(
                384
            ),

            nn.Linear(
                384,
                512
            ),

            nn.ReLU(),

            nn.Dropout(
                dropout
            ),

            nn.Linear(
                512,
                256
            ),

            nn.ReLU(),

            nn.Dropout(
                dropout
            ),

            nn.Linear(
                256,
                num_classes
            )
        )


    def classify_pairs(
        self,
        z,
        pairs
    ):

        z1 = z[
            pairs[:, 0]
        ]

        z2 = z[
            pairs[:, 1]
        ]


        ordered_concat_z1_z2 = torch.cat(
            [
                z1,
                z2
            ],
            dim=1
        )


        return self.classifier(
            ordered_concat_z1_z2
        )


# ============================================================
# 18) METRICS
# ============================================================

def calculate_metrics(
    y_true,
    y_pred
):

    return {

        "accuracy":
            float(
                accuracy_score(
                    y_true,
                    y_pred
                )
            ),

        "macro84":
            float(
                f1_score(
                    y_true,
                    y_pred,
                    labels=np.arange(
                        NUM_CLASSES
                    ),
                    average="macro",
                    zero_division=0
                )
            ),

        "weighted_f1":
            float(
                f1_score(
                    y_true,
                    y_pred,
                    labels=np.arange(
                        NUM_CLASSES
                    ),
                    average="weighted",
                    zero_division=0
                )
            )
    }


# ============================================================
# 19) EVALUATION
# ============================================================

@torch.no_grad()
def evaluate_split(
    model,
    x,
    edge_index,
    pairs,
    y_true
):

    model.eval()


    z = model.encoder(
        x,
        edge_index
    )


    pred_chunks = []


    for start in range(
        0,
        len(pairs),
        BATCH_SIZE
    ):

        end = min(
            start + BATCH_SIZE,
            len(pairs)
        )


        logits = model.classify_pairs(

            z,

            pairs[
                start:end
            ]
        )


        pred = (
            logits
            .argmax(
                dim=1
            )
            .detach()
            .cpu()
            .numpy()
        )


        pred_chunks.append(
            pred
        )


    y_pred = np.concatenate(
        pred_chunks
    )


    metrics = calculate_metrics(
        y_true,
        y_pred
    )


    return (
        metrics,
        y_pred
    )


# ============================================================
# 20) SAVE BEST MODEL
# ============================================================

def save_best_checkpoint(
    path,
    model,
    seed,
    epoch,
    selection_criterion,
    cal_metrics
):

    payload = {

        "seed":
            int(
                seed
            ),

        "epoch":
            int(
                epoch
            ),

        "selection_criterion":
            selection_criterion,

        "cal_metrics":
            {
                key:
                    float(
                        value
                    )

                for key, value

                in cal_metrics.items()
            },

        "model_state":
            {
                key:
                    tensor.detach()
                    .cpu()
                    .clone()

                for key, tensor

                in model.state_dict().items()
            }
    }


    torch.save(
        payload,
        path
    )


# ============================================================
# 21) TRAIN ONE SEED
# ============================================================

def run_seed(seed):

    print(
        "\n"
        +
        "#" * 100
    )

    print(
        f"MODEL-SELECTION SENSITIVITY — SEED {seed}"
    )

    print(
        "#" * 100
    )


    seed_dir = (
        OUT_ROOT
        /
        f"seed_{seed}"
    )


    seed_dir.mkdir(
        parents=True,
        exist_ok=True
    )


    final_json = (
        seed_dir
        /
        "final_result.json"
    )


    # --------------------------------------------------------
    # Already finished
    # --------------------------------------------------------

    if final_json.is_file():

        print(
            f" Seed {seed} already complete — no retraining."
        )

        return json.loads(
            final_json.read_text(
                encoding="utf-8"
            )
        )


    set_seed(
        seed
    )


    model = FinalConcatOnlyGCN(

        input_dim=INPUT_DIM,

        num_classes=NUM_CLASSES,

        dropout=DROPOUT

    ).to(
        device
    )


    total_parameters = sum(

        parameter.numel()

        for parameter

        in model.parameters()
    )


    criterion = nn.CrossEntropyLoss(
        weight=class_weights_t
    )


    optimizer = torch.optim.AdamW(

        model.parameters(),

        lr=LR,

        weight_decay=WEIGHT_DECAY
    )


    scheduler = (
        torch.optim.lr_scheduler
        .ReduceLROnPlateau(

            optimizer,

            mode="max",

            factor=0.5,

            patience=5
        )
    )


    # --------------------------------------------------------
    # Three model-selection checkpoints
    # --------------------------------------------------------

    checkpoint_paths = {

        "macro_f1":
            seed_dir
            /
            "best_by_macro_f1.pt",

        "accuracy":
            seed_dir
            /
            "best_by_accuracy.pt",

        "weighted_f1":
            seed_dir
            /
            "best_by_weighted_f1.pt"
    }


    resume_path = (
        seed_dir
        /
        "last_training_state.pt"
    )


    history_path = (
        seed_dir
        /
        "training_history.csv"
    )


    best_values = {

        "macro_f1":
            -np.inf,

        "accuracy":
            -np.inf,

        "weighted_f1":
            -np.inf
    }


    best_epochs = {

        "macro_f1":
            None,

        "accuracy":
            None,

        "weighted_f1":
            None
    }


    history = []

    start_epoch = 1

    epochs_without_macro_improvement = 0


    # ========================================================
    # RESUME
    # ========================================================

    if resume_path.is_file():

        resume = torch.load(
            resume_path,
            map_location="cpu",
            weights_only=False
        )


        model.load_state_dict(
            resume[
                "model_state"
            ]
        )


        model.to(
            device
        )


        optimizer.load_state_dict(
            resume[
                "optimizer_state"
            ]
        )


        for optimizer_state in optimizer.state.values():

            for key, value in optimizer_state.items():

                if torch.is_tensor(
                    value
                ):

                    optimizer_state[
                        key
                    ] = value.to(
                        device
                    )


        scheduler.load_state_dict(
            resume[
                "scheduler_state"
            ]
        )


        best_values = resume[
            "best_values"
        ]


        best_epochs = resume[
            "best_epochs"
        ]


        history = resume[
            "history"
        ]


        epochs_without_macro_improvement = resume[
            "epochs_without_macro_improvement"
        ]


        start_epoch = (
            int(
                resume[
                    "epoch"
                ]
            )
            +
            1
        )


        set_rng_state(
            resume[
                "rng_state"
            ]
        )


        print(
            f"↻ Resuming from epoch {start_epoch}"
        )


    # ========================================================
    # TRAINING
    # ========================================================

    for epoch in range(
        start_epoch,
        MAX_EPOCHS + 1
    ):

        model.train()


        permutation = torch.randperm(
            len(
                train_pairs_t
            ),
            device=device
        )


        total_loss = 0.0

        total_examples = 0


        for start in range(
            0,
            len(
                train_pairs_t
            ),
            BATCH_SIZE
        ):

            batch_indices = permutation[
                start:
                start + BATCH_SIZE
            ]


            optimizer.zero_grad(
                set_to_none=True
            )


            # Same controlled trainer:
            # full Train graph encoded per pair minibatch.
            z = model.encoder(
                x_train,
                edge_train
            )


            logits = model.classify_pairs(

                z,

                train_pairs_t[
                    batch_indices
                ]
            )


            loss = criterion(

                logits,

                y_train_t[
                    batch_indices
                ]
            )


            loss.backward()


            optimizer.step()


            batch_size_actual = len(
                batch_indices
            )


            total_loss += (
                float(
                    loss
                    .detach()
                    .cpu()
                    .item()
                )
                *
                batch_size_actual
            )


            total_examples += (
                batch_size_actual
            )


        train_loss = (
            total_loss
            /
            total_examples
        )


        # ====================================================
        # CALIBRATION EVALUATION ONLY
        # ====================================================

        cal_metrics, _ = evaluate_split(

            model=model,

            x=x_all,

            edge_index=edge_global,

            pairs=cal_pairs_t,

            y_true=y_cal
        )


        cal_macro = cal_metrics[
            "macro84"
        ]


        cal_accuracy = cal_metrics[
            "accuracy"
        ]


        cal_weighted = cal_metrics[
            "weighted_f1"
        ]


        # Scheduler remains baseline Macro-F1 driven
        scheduler.step(
            cal_macro
        )


        current_lr = optimizer.param_groups[
            0
        ][
            "lr"
        ]


        # ====================================================
        # BEST BY MACRO-F1
        # ====================================================

        macro_improved = (
            cal_macro
            >
            best_values[
                "macro_f1"
            ]
        )


        if macro_improved:

            best_values[
                "macro_f1"
            ] = cal_macro


            best_epochs[
                "macro_f1"
            ] = epoch


            save_best_checkpoint(

                path=checkpoint_paths[
                    "macro_f1"
                ],

                model=model,

                seed=seed,

                epoch=epoch,

                selection_criterion=(
                    "calibration_macro_f1"
                ),

                cal_metrics=cal_metrics
            )


            epochs_without_macro_improvement = 0


        else:

            epochs_without_macro_improvement += 1


        # ====================================================
        # BEST BY ACCURACY
        # ====================================================

        if (
            cal_accuracy
            >
            best_values[
                "accuracy"
            ]
        ):

            best_values[
                "accuracy"
            ] = cal_accuracy


            best_epochs[
                "accuracy"
            ] = epoch


            save_best_checkpoint(

                path=checkpoint_paths[
                    "accuracy"
                ],

                model=model,

                seed=seed,

                epoch=epoch,

                selection_criterion=(
                    "calibration_accuracy"
                ),

                cal_metrics=cal_metrics
            )


        # ====================================================
        # BEST BY WEIGHTED-F1
        # ====================================================

        if (
            cal_weighted
            >
            best_values[
                "weighted_f1"
            ]
        ):

            best_values[
                "weighted_f1"
            ] = cal_weighted


            best_epochs[
                "weighted_f1"
            ] = epoch


            save_best_checkpoint(

                path=checkpoint_paths[
                    "weighted_f1"
                ],

                model=model,

                seed=seed,

                epoch=epoch,

                selection_criterion=(
                    "calibration_weighted_f1"
                ),

                cal_metrics=cal_metrics
            )


        # ====================================================
        # HISTORY
        # ====================================================

        row = {

            "epoch":
                epoch,

            "train_loss":
                train_loss,

            "lr":
                current_lr,

            "cal_accuracy":
                cal_accuracy,

            "cal_macro84":
                cal_macro,

            "cal_weighted_f1":
                cal_weighted,

            "best_macro_epoch":
                best_epochs[
                    "macro_f1"
                ],

            "best_accuracy_epoch":
                best_epochs[
                    "accuracy"
                ],

            "best_weighted_epoch":
                best_epochs[
                    "weighted_f1"
                ],

            "epochs_without_macro_improvement":
                epochs_without_macro_improvement
        }


        history.append(
            row
        )


        pd.DataFrame(
            history
        ).to_csv(
            history_path,
            index=False
        )


        print(

            f"Seed {seed} | "

            f"Epoch {epoch:03d} | "

            f"Loss {train_loss:.6f} | "

            f"CAL Acc {cal_accuracy:.6f} | "

            f"Macro84 {cal_macro:.6f} | "

            f"Weighted {cal_weighted:.6f} | "

            f"LR {current_lr:.2e} | "

            f"Best epochs "
            f"M={best_epochs['macro_f1']} "
            f"A={best_epochs['accuracy']} "
            f"W={best_epochs['weighted_f1']}"
        )


        # ====================================================
        # AUTO-RESUME SAVE
        # ====================================================

        resume_payload = {

            "seed":
                seed,

            "epoch":
                epoch,

            "model_state":
                model.state_dict(),

            "optimizer_state":
                optimizer.state_dict(),

            "scheduler_state":
                scheduler.state_dict(),

            "best_values":
                best_values,

            "best_epochs":
                best_epochs,

            "history":
                history,

            "epochs_without_macro_improvement":
                epochs_without_macro_improvement,

            "rng_state":
                get_rng_state()
        }


        torch.save(
            resume_payload,
            resume_path
        )


        # ====================================================
        # BASELINE EARLY STOP RULE
        # ====================================================

        if (
            epochs_without_macro_improvement
            >=
            EARLY_STOP_PATIENCE
        ):

            print(
                "\nEarly stopping at epoch",
                epoch,
                "based on Calibration Macro-F1."
            )

            break


    completed_epochs = (
        history[
            -1
        ][
            "epoch"
        ]
    )


    # ========================================================
    # TEST ONLY AFTER TRAINING / MODEL SELECTION
    # ========================================================

    print(
        "\n"
        +
        "=" * 100
    )

    print(
        "FINAL TEST EVALUATION"
    )

    print(
        "=" * 100
    )

    print(
        "Training finished."
    )

    print(
        "Evaluating only the three "
        "Calibration-selected checkpoints."
    )


    selection_results = {}


    selection_specs = [

        (
            "macro_f1",
            "Calibration Macro-F1"
        ),

        (
            "accuracy",
            "Calibration Accuracy"
        ),

        (
            "weighted_f1",
            "Calibration Weighted-F1"
        )
    ]


    for criterion_key, display_name in selection_specs:

        checkpoint_path = checkpoint_paths[
            criterion_key
        ]


        assert checkpoint_path.is_file()


        checkpoint = torch.load(
            checkpoint_path,
            map_location="cpu",
            weights_only=False
        )


        model.load_state_dict(
            checkpoint[
                "model_state"
            ]
        )


        model.to(
            device
        )


        # ----------------------------------------------------
        # Reloaded Calibration QA
        # ----------------------------------------------------

        reloaded_cal_metrics, _ = evaluate_split(

            model=model,

            x=x_all,

            edge_index=edge_global,

            pairs=cal_pairs_t,

            y_true=y_cal
        )


        saved_cal_metrics = checkpoint[
            "cal_metrics"
        ]


        cal_gap = max(

            abs(
                reloaded_cal_metrics[
                    "accuracy"
                ]
                -
                saved_cal_metrics[
                    "accuracy"
                ]
            ),

            abs(
                reloaded_cal_metrics[
                    "macro84"
                ]
                -
                saved_cal_metrics[
                    "macro84"
                ]
            ),

            abs(
                reloaded_cal_metrics[
                    "weighted_f1"
                ]
                -
                saved_cal_metrics[
                    "weighted_f1"
                ]
            )
        )


        assert cal_gap <= 1e-10, (
            f"Calibration reload gap too large: {cal_gap}"
        )


        # ----------------------------------------------------
        # TEST
        # ----------------------------------------------------

        test_metrics, test_predictions = evaluate_split(

            model=model,

            x=x_all,

            edge_index=edge_global,

            pairs=test_pairs_t,

            y_true=y_test
        )


        prediction_file = (
            seed_dir
            /
            (
                f"test_predictions_"
                f"best_by_{criterion_key}.npy"
            )
        )


        np.save(
            prediction_file,
            test_predictions
        )


        selection_results[
            criterion_key
        ] = {

            "selection_name":
                display_name,

            "selected_epoch":
                int(
                    checkpoint[
                        "epoch"
                    ]
                ),

            "saved_cal_accuracy":
                float(
                    saved_cal_metrics[
                        "accuracy"
                    ]
                ),

            "saved_cal_macro84":
                float(
                    saved_cal_metrics[
                        "macro84"
                    ]
                ),

            "saved_cal_weighted_f1":
                float(
                    saved_cal_metrics[
                        "weighted_f1"
                    ]
                ),

            "reloaded_cal_accuracy":
                float(
                    reloaded_cal_metrics[
                        "accuracy"
                    ]
                ),

            "reloaded_cal_macro84":
                float(
                    reloaded_cal_metrics[
                        "macro84"
                    ]
                ),

            "reloaded_cal_weighted_f1":
                float(
                    reloaded_cal_metrics[
                        "weighted_f1"
                    ]
                ),

            "calibration_reload_gap":
                float(
                    cal_gap
                ),

            "test_accuracy":
                float(
                    test_metrics[
                        "accuracy"
                    ]
                ),

            "test_macro84":
                float(
                    test_metrics[
                        "macro84"
                    ]
                ),

            "test_weighted_f1":
                float(
                    test_metrics[
                        "weighted_f1"
                    ]
                ),

            "checkpoint_sha256":
                sha256_file(
                    checkpoint_path
                ),

            "test_predictions_sha256":
                sha256_file(
                    prediction_file
                )
        }


        print(
            "\n"
            +
            "-" * 90
        )

        print(
            display_name
        )

        print(
            "-" * 90
        )

        print(
            "Selected epoch:",
            checkpoint[
                "epoch"
            ]
        )

        print(
            "CAL Accuracy:",
            reloaded_cal_metrics[
                "accuracy"
            ]
        )

        print(
            "CAL Macro84:",
            reloaded_cal_metrics[
                "macro84"
            ]
        )

        print(
            "CAL Weighted-F1:",
            reloaded_cal_metrics[
                "weighted_f1"
            ]
        )

        print(
            "TEST Accuracy:",
            test_metrics[
                "accuracy"
            ]
        )

        print(
            "TEST Macro84:",
            test_metrics[
                "macro84"
            ]
        )

        print(
            "TEST Weighted-F1:",
            test_metrics[
                "weighted_f1"
            ]
        )


    # ========================================================
    # FINAL SEED RESULT
    # ========================================================

    result = {

        "experiment":
            "checkpoint_selection_sensitivity",

        "controlled_factor":
            "checkpoint_selection_criterion",

        "feature_mode":
            "morgan_rdkit",

        "seed":
            int(
                seed
            ),

        "input_dim":
            2249,

        "num_classes":
            84,

        "model":
            "concat_only_gcn",

        "graph":
            "TRAIN_ORDERED_UNIQUE_THEN_REVERSE",

        "graph_forward_unique_edges":
            134122,

        "graph_total_edges":
            268244,

        "graph_reciprocal_overlap":
            122,

        "train_rows":
            134249,

        "calibration_rows":
            28768,

        "test_rows":
            28768,

        "train_fit_drugs":
            1675,

        "total_drugs":
            1705,

        "calibration_unseen_drugs":
            14,

        "test_unseen_drugs":
            20,

        "training_early_stopping_criterion":
            "calibration_macro_f1",

        "scheduler_criterion":
            "calibration_macro_f1",

        "early_stopping_patience":
            18,

        "max_epochs":
            120,

        "completed_epochs":
            int(
                completed_epochs
            ),

        "best_epochs":
            {
                key:
                    int(
                        value
                    )

                for key, value

                in best_epochs.items()
            },

        "total_parameters":
            int(
                total_parameters
            ),

        "selection_results":
            selection_results,

        "feature_cache_sha256":
            sha256_file(
                FEATURE_FILE
            )
    }


    final_json.write_text(

        json.dumps(
            result,
            indent=2
        ),

        encoding="utf-8"
    )


    # ========================================================
    # PER-SEED ZIP
    # ========================================================

    seed_zip = (
        seed_dir
        /
        (
            f"MODEL_SELECTION_SENSITIVITY_"
            f"SEED_{seed}_COMPLETE.zip"
        )
    )


    with zipfile.ZipFile(

        seed_zip,

        "w",

        compression=zipfile.ZIP_DEFLATED

    ) as archive:

        for path in seed_dir.iterdir():

            if (
                path.is_file()
                and
                path != seed_zip
            ):

                archive.write(
                    path,
                    arcname=path.name
                )


    with zipfile.ZipFile(
        seed_zip,
        "r"
    ) as archive:

        assert archive.testzip() is None


    print(
        "\n"
        +
        "=" * 100
    )

    print(
        f" SEED {seed} COMPLETE"
    )

    print(
        "=" * 100
    )

    print(
        "Macro-F1 best epoch:",
        best_epochs[
            "macro_f1"
        ]
    )

    print(
        "Accuracy best epoch:",
        best_epochs[
            "accuracy"
        ]
    )

    print(
        "Weighted-F1 best epoch:",
        best_epochs[
            "weighted_f1"
        ]
    )

    print(
        "\nZIP:"
    )

    print(
        seed_zip
    )


    del model

    gc.collect()

    if torch.cuda.is_available():

        torch.cuda.empty_cache()


    return result


# ============================================================
# 22) RUN ALL 3 SEEDS
# ============================================================

all_results = []


for seed in SEEDS:

    result = run_seed(
        seed
    )

    all_results.append(
        result
    )


# ============================================================
# 23) LONG RESULTS TABLE
# ============================================================

rows = []


for result in all_results:

    seed = result[
        "seed"
    ]


    for criterion_key, values in result[
        "selection_results"
    ].items():

        rows.append({

            "seed":
                seed,

            "criterion":
                criterion_key,

            "selection_name":
                values[
                    "selection_name"
                ],

            "selected_epoch":
                values[
                    "selected_epoch"
                ],

            "cal_accuracy":
                values[
                    "reloaded_cal_accuracy"
                ],

            "cal_macro84":
                values[
                    "reloaded_cal_macro84"
                ],

            "cal_weighted_f1":
                values[
                    "reloaded_cal_weighted_f1"
                ],

            "test_accuracy":
                values[
                    "test_accuracy"
                ],

            "test_macro84":
                values[
                    "test_macro84"
                ],

            "test_weighted_f1":
                values[
                    "test_weighted_f1"
                ]
        })


long_df = pd.DataFrame(
    rows
).sort_values(
    [
        "criterion",
        "seed"
    ]
).reset_index(
    drop=True
)


LONG_CSV = (
    OUT_ROOT
    /
    "all_seed_checkpoint_selection_results.csv"
)


long_df.to_csv(
    LONG_CSV,
    index=False
)


# ============================================================
# 24) MEAN ± SAMPLE SD
# ============================================================

summary_rows = []


criterion_order = [
    "macro_f1",
    "accuracy",
    "weighted_f1"
]


for criterion_key in criterion_order:

    group = (
        long_df[
            long_df[
                "criterion"
            ]
            ==
            criterion_key
        ]
        .copy()
    )


    record = {

        "criterion":
            criterion_key,

        "selection_name":
            group[
                "selection_name"
            ]
            .iloc[
                0
            ],

        "n_seeds":
            len(
                group
            )
    }


    for metric in [

        "selected_epoch",

        "cal_accuracy",

        "cal_macro84",

        "cal_weighted_f1",

        "test_accuracy",

        "test_macro84",

        "test_weighted_f1"

    ]:

        values = (
            group[
                metric
            ]
            .to_numpy(
                dtype=float
            )
        )


        mean_value = float(
            values.mean()
        )


        sd_value = float(
            values.std(
                ddof=1
            )
        )


        record[
            f"{metric}_mean"
        ] = mean_value


        record[
            f"{metric}_sd"
        ] = sd_value


        if metric != "selected_epoch":

            record[
                f"{metric}_paper"
            ] = (
                f"{mean_value * 100:.4f} "
                f"± "
                f"{sd_value * 100:.4f}%"
            )


    summary_rows.append(
        record
    )


summary_df = pd.DataFrame(
    summary_rows
)


SUMMARY_CSV = (
    OUT_ROOT
    /
    "checkpoint_selection_mean_sd.csv"
)


summary_df.to_csv(
    SUMMARY_CSV,
    index=False
)


# ============================================================
# 25) MATCHED-SEED DIFFERENCES VS MACRO-F1
# ============================================================

macro_df = (
    long_df[
        long_df[
            "criterion"
        ]
        ==
        "macro_f1"
    ]
    .set_index(
        "seed"
    )
)


difference_rows = []


for alternative in [
    "accuracy",
    "weighted_f1"
]:

    alternative_df = (
        long_df[
            long_df[
                "criterion"
            ]
            ==
            alternative
        ]
        .set_index(
            "seed"
        )
    )


    for metric in [

        "test_accuracy",

        "test_macro84",

        "test_weighted_f1"

    ]:

        delta = (
            alternative_df[
                metric
            ]
            -
            macro_df[
                metric
            ]
        )


        difference_rows.append({

            "alternative_criterion":
                alternative,

            "reference_criterion":
                "macro_f1",

            "metric":
                metric,

            "seed42_delta_pp":
                float(
                    delta.loc[
                        42
                    ]
                    *
                    100
                ),

            "seed43_delta_pp":
                float(
                    delta.loc[
                        43
                    ]
                    *
                    100
                ),

            "seed44_delta_pp":
                float(
                    delta.loc[
                        44
                    ]
                    *
                    100
                ),

            "mean_delta_pp":
                float(
                    delta.mean()
                    *
                    100
                ),

            "sd_delta_pp":
                float(
                    delta.std(
                        ddof=1
                    )
                    *
                    100
                )
        })


difference_df = pd.DataFrame(
    difference_rows
)


DIFFERENCE_CSV = (
    OUT_ROOT
    /
    "matched_seed_differences_vs_macro.csv"
)


difference_df.to_csv(
    DIFFERENCE_CSV,
    index=False
)


# ============================================================
# 26) DISPLAY RESULTS
# ============================================================

print(
    "\n"
    +
    "=" * 100
)

print(
    " MODEL-SELECTION SENSITIVITY COMPLETE"
)

print(
    "=" * 100
)


print(
    "\nPER-SEED RESULTS"
)


display(

    long_df[
        [
            "seed",
            "selection_name",
            "selected_epoch",
            "cal_accuracy",
            "cal_macro84",
            "cal_weighted_f1",
            "test_accuracy",
            "test_macro84",
            "test_weighted_f1"
        ]
    ]
)


print(
    "\n"
    +
    "=" * 100
)

print(
    "MEAN ± SAMPLE SD"
)

print(
    "=" * 100
)


display(

    summary_df[
        [
            "selection_name",
            "selected_epoch_mean",
            "selected_epoch_sd",
            "test_accuracy_paper",
            "test_macro84_paper",
            "test_weighted_f1_paper"
        ]
    ]
)


print(
    "\n"
    +
    "=" * 100
)

print(
    "MATCHED-SEED DIFFERENCES VS MACRO-F1"
)

print(
    "=" * 100
)


display(
    difference_df
)


# ============================================================
# 27) FINAL 3-SEED ZIP
# ============================================================

FINAL_ZIP = Path(
    "/kaggle/working/"
    "MODEL_SELECTION_SENSITIVITY_3SEEDS.zip"
)


with zipfile.ZipFile(

    FINAL_ZIP,

    "w",

    compression=zipfile.ZIP_DEFLATED

) as archive:


    archive.write(
        LONG_CSV,
        arcname=LONG_CSV.name
    )


    archive.write(
        SUMMARY_CSV,
        arcname=SUMMARY_CSV.name
    )


    archive.write(
        DIFFERENCE_CSV,
        arcname=DIFFERENCE_CSV.name
    )


    for seed in SEEDS:

        seed_dir = (
            OUT_ROOT
            /
            f"seed_{seed}"
        )


        for path in seed_dir.iterdir():

            if path.is_file():

                archive.write(

                    path,

                    arcname=(
                        f"seed_{seed}/"
                        f"{path.name}"
                    )
                )


with zipfile.ZipFile(
    FINAL_ZIP,
    "r"
) as archive:

    assert archive.testzip() is None


# ============================================================
# 28) FINAL QA
# ============================================================

print(
    "\n"
    +
    "=" * 100
)

print(
    " FINAL MODEL-SELECTION PACKAGE READY"
)

print(
    "=" * 100
)


print(
    "Results:"
)

print(
    LONG_CSV
)


print(
    "\nMean ± SD:"
)

print(
    SUMMARY_CSV
)


print(
    "\nMatched-seed differences:"
)

print(
    DIFFERENCE_CSV
)


print(
    "\nZIP:"
)

print(
    FINAL_ZIP
)


print(
    "\nQA:"
)

print(
    " Morgan + RDKit = 2249"
)

print(
    " Seeds = 42 / 43 / 44"
)

print(
    " 84 Interaction classes unchanged"
)

print(
    " Ordered Drug1→Drug2 preserved"
)

print(
    " Unique forward Train edges = 134,122"
)

print(
    " Final Train graph edges = 268,244"
)

print(
    " Macro-F1 checkpoint saved"
)

print(
    " Accuracy checkpoint saved"
)

print(
    " Weighted-F1 checkpoint saved"
)

print(
    " Same training trajectory within each seed"
)

print(
    " Scheduler remains Macro-F1 driven"
)

print(
    " Early stopping remains Macro-F1 driven"
)

print(
    " Test evaluated only after model selection"
)

print(
    " Matched-seed comparison generated"
)

print(
    " ZIP integrity passed"
)